# 17 — SQLite Transactions and Data Integrity

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
import sqlite3

from lesson import Exercises, ___, run

check = Exercises('advanced/17', {1: '6d5250ae567b1245', 2: '9ac97ad97efcd59c', 3: 'e06c04a4ad326b32', 4: '6914bbe69b36a5f8', 5: 'ea3cc304ef01cd8e', 6: 'c0edd9728ea633f4', 7: 'af2fa5634585f46c'}, practice='advanced_03_security_testing_performance', independent='add_users')

## 1. Atomic changes

- Changes that belong together go in one transaction: `with conn:` commits all or rolls back all.
- `executemany` inserts many rows in one statement; inside `with conn:` it's one transaction.
- A failure partway through leaves nothing half-done.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (email TEXT UNIQUE)")
try:
    with conn:
        conn.executemany("INSERT INTO users VALUES (?)", [("a@x",), ("b@x",), ("a@x",)])
except sqlite3.IntegrityError:
    print("duplicate, rolled back")
print(conn.execute("SELECT COUNT(*) FROM users").fetchone()[0])

**Exercise 1.** After the failed batch in the demo, how many users are stored?

In [ ]:
predicted = ___
check(1, predicted)

**Exercise 2.** Insert a clean batch in one transaction and count the rows.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (email TEXT UNIQUE)")
with conn:
    conn.executemany("INSERT INTO users VALUES (?)", [(e,) for e in ___])
check(2, conn.execute("SELECT COUNT(*) FROM users").fetchone()[0])   # insert 3 emails

## 2. Constraints, not check-then-insert

- "Check if it exists, then insert" races: two requests can both pass the check.
- Let the database enforce rules with `UNIQUE`, `NOT NULL`, `CHECK`, `FOREIGN KEY`, and catch `IntegrityError`.
- Turn the database error into a domain error (`ValueError("email already registered")`).

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (email TEXT UNIQUE NOT NULL)")
def register(email):
    try:
        with conn:
            conn.execute("INSERT INTO users VALUES (?)", (email,))
    except sqlite3.IntegrityError as error:
        raise ValueError("email already registered") from error

register("a@x")
try:
    register("a@x")
except ValueError as error:
    print(error)

**Exercise 3.** Which constraint stops two rows with the same email? Write the SQL keyword.

In [ ]:
answer = ___
check(3, answer)

**Exercise 4.** Which `sqlite3` exception does a constraint violation raise? Write its class name.

In [ ]:
answer = ___
check(4, answer)

**Exercise 5.** Reserve all keys in slots(name TEXT UNIQUE NOT NULL). Use the supplied connection. Return True on success. On a constraint violation roll back the entire batch and return False. Preserve previously committed reservations.

In [ ]:
def reserve(db, keys):
    raise NotImplementedError

def observe(save):
    import sqlite3
    from contextlib import closing
    with closing(sqlite3.connect(":memory:")) as db:
        db.execute("CREATE TABLE slots(name TEXT UNIQUE NOT NULL)")
        outcomes = [save(db, ["a"]), save(db, ["x", "a"]), save(db, ["b", None]), save(db, [])]
        return outcomes, db.execute("SELECT name FROM slots ORDER BY name").fetchall()

check(5, reserve, observe=observe)

## 3. Cleanup

- `with conn:` manages the transaction but does not close the connection. Close it with `conn.close()` or `contextlib.closing(conn)`.
- Turn on foreign keys per connection: `PRAGMA foreign_keys = ON`.
- Use `:memory:` or a temp file in tests.

In [ ]:
import sqlite3
from contextlib import closing
with closing(sqlite3.connect(":memory:")) as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    print(conn.execute("PRAGMA foreign_keys").fetchone())

**Exercise 6.** Does `with conn:` close the connection? Write `True` or `False`.

In [ ]:
answer = ___
check(6, answer)

**Exercise 7.** Which `contextlib` helper closes the connection at the end of a `with`? Write its name.

In [ ]:
answer = ___
check(7, answer)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### Independent build — `add_users`

Insert the whole email batch into users(email UNIQUE) atomically. Each address must have nonempty local, domain and suffix parts, a single @ and no whitespace. Invalid or duplicate entries raise ValueError and leave existing rows intact.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def add_users(conn: sqlite3.Connection, emails: list[str]) -> None:
    raise NotImplementedError

In [ ]:
check('add_users')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Why enforce uniqueness in the database, not just Python?

   - **a)** Python can't compare strings
   - **b)** Concurrent writers can bypass Python checks
   - **c)** It's prettier

**2.** At-least-once delivery means a message may be…

   - **a)** Encrypted
   - **b)** Processed more than once
   - **c)** Lost

**3.** To make processing effectively exactly-once, combine at-least-once with…

   - **a)** Idempotent writes (e.g. unique keys)
   - **b)** Bigger servers
   - **c)** Retries

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/17', {1: '2d20b504f349', 2: '22cdefbd6b25', 3: 'ba459d8366c4'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [18 — Production Quality: Packaging, CI, Configuration, and Deployment](18_production_quality_ci_deployment.ipynb)